# Noise2Void Prediction

Load a trained 2D or 3D Noise2Void model, predict on a CZI nuclei volume, and compare the result.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import napari
import numpy as np
from czifile import imread as czi_imread
from n2v.models import N2V
from tifffile import imwrite

In [7]:
MODEL_DIR = Path('models')
MODEL_NAME = 'n2v_nuclei_3d_all'
MODEL_MODE = '3d'   # '2d' or '3d'
CHANNEL = 2

IMAGE_PATH = Path(
"DATA/eGFP-LaminB3/ST22/260304_ID468_1024x1024_1zoom_1umz__ST22_GFPlaminB3_laminB1ab_embryo2_1-5zoom_-01.czi"
)

SLICE_INDEX = None
N_TILES_2D = (2, 2)
N_TILES_3D = (1, 2, 2, 1)

In [8]:
img = czi_imread(IMAGE_PATH)
volume = img[0, 0, 0, :, :, :, 0]

In [6]:
import napari
viewer = napari.Viewer()
viewer.add_image(volume)

<Image layer 'volume' at 0x24c362f5790>

In [7]:
def load_nuclei_volume(image_path):
    image = czi_imread(str(image_path))
    volume = image[0, 0, CHANNEL, 0, :, :, :, 0]
    volume = np.asarray(volume, dtype=np.float32)
    volume = np.squeeze(volume)
    if volume.ndim == 2:
        volume = volume[np.newaxis, ...]
    return volume


def percentile_normalize(img, pmin=1.0, pmax=99.8):
    lo = np.percentile(img, pmin)
    hi = np.percentile(img, pmax)
    if hi <= lo:
        return np.zeros_like(img, dtype=np.float32)
    img = np.clip(img, lo, hi)
    return ((img - lo) / (hi - lo)).astype(np.float32)


def rescale_for_display(img, pmin=1.0, pmax=99.8):
    lo = np.percentile(img, pmin)
    hi = np.percentile(img, pmax)
    if hi <= lo:
        return np.zeros_like(img, dtype=np.float32)
    img = np.clip(img, lo, hi)
    return ((img - lo) / (hi - lo)).astype(np.float32)

In [8]:
def predict_2d(model, volume):
    pred_slices = []
    for z in range(volume.shape[0]):
        pred = model.predict(volume[z], axes='YX', n_tiles=N_TILES_2D)
        pred_slices.append(pred.astype(np.float32))
    return np.stack(pred_slices, axis=0)


def predict_3d(model, volume):
    volume_zyxc = volume[..., np.newaxis]
    pred = model.predict(volume_zyxc, axes='ZYXC', n_tiles=N_TILES_3D)
    return np.squeeze(pred, axis=-1).astype(np.float32)

In [9]:
model = N2V(config=None, name=MODEL_NAME, basedir=str(MODEL_DIR))
img_nuclei = load_nuclei_volume(IMAGE_PATH)
img_nuclei_norm = percentile_normalize(img_nuclei)

if MODEL_MODE == '2d':
    img_nuclei_pred = predict_2d(model, img_nuclei_norm)
elif MODEL_MODE == '3d':
    img_nuclei_pred = predict_3d(model, img_nuclei_norm)
else:
    raise ValueError("MODEL_MODE must be '2d' or '3d'")

img_nuclei_pred_disp = rescale_for_display(img_nuclei_pred)

print('mode         :', MODEL_MODE)
print('input shape  :', img_nuclei.shape)
print('pred shape   :', img_nuclei_pred.shape)
print('input range  :', float(img_nuclei_norm.min()), float(img_nuclei_norm.max()))
print('pred range   :', float(img_nuclei_pred.min()), float(img_nuclei_pred.max()))
print('model path   :', MODEL_DIR / MODEL_NAME)
print('image path   :', IMAGE_PATH)

Loading network weights from 'weights_best.h5'.
1/1 [==============================] - 12s 12s/step


 25%|██▌       | 1/4 [00:00<?, ?it/s]

1/1 [==============================] - 1s 1s/step


 50%|█████     | 2/4 [00:01<00:01,  1.32it/s]

1/1 [==============================] - 0s 414ms/step


 75%|███████▌  | 3/4 [00:03<00:01,  1.07s/it]

1/1 [==============================] - 0s 412ms/step


100%|██████████| 4/4 [00:04<00:00,  1.16s/it]


mode         : 3d
input shape  : (52, 1024, 1024)
pred shape   : (52, 1024, 1024)
input range  : 0.0 1.0
pred range   : 0.00638270378112793 1.109816551208496
model path   : models\n2v_nuclei_3d_all
image path   : DATA\eGFP-LaminB3\ST22\260304_ID468_1024x1024_1zoom_1umz__ST22_GFPlaminB3_laminB1ab_embryo3_1-5zoom_-01.czi


In [10]:
viewer = napari.Viewer()
viewer.add_image(img_nuclei_norm, name='nuclei_norm', contrast_limits=(0, 1))
viewer.add_image(img_nuclei_pred_disp, name=f'nuclei_denoised_{MODEL_MODE}', contrast_limits=(0, 1))
viewer.add_image(img_nuclei_norm - img_nuclei_pred_disp, name='residual')

<Image layer 'residual' at 0x24247810700>

In [12]:
all_paths = sorted(Path("DATA/eGFP-LaminB3").rglob("*.czi"))
print("files:", len(all_paths))

out_dir = Path("denoised_n2v")
out_dir.mkdir(exist_ok=True)

model = N2V(config=None, name=MODEL_NAME, basedir=str(MODEL_DIR))

for image_path in all_paths:
    print("processing:", image_path.name)

    img_nuclei = load_nuclei_volume(image_path)
    img_nuclei_norm = percentile_normalize(img_nuclei)

    if MODEL_MODE == "2d":
        img_nuclei_pred = predict_2d(model, img_nuclei_norm)
    elif MODEL_MODE == "3d":
        img_nuclei_pred = predict_3d(model, img_nuclei_norm)
    else:
        raise ValueError("MODEL_MODE must be '2d' or '3d'")

    save_name = image_path.stem + f"_n2v_nuclei_{MODEL_MODE}.tif"
    imwrite(out_dir / save_name, img_nuclei_pred.astype(np.float32))


files: 7
Loading network weights from 'weights_best.h5'.
processing: 260304_ID468_1024x1024_1zoom_1umz__ST22_GFPlaminB3_laminB1ab_embryo2_1-5zoom_-01.czi
1/1 [==============================] - 5s 5s/step


 25%|██▌       | 1/4 [00:00<?, ?it/s]

1/1 [==============================] - 1s 588ms/step


 50%|█████     | 2/4 [00:02<00:02,  1.05s/it]

1/1 [==============================] - 1s 665ms/step


...

 50%|█████     | 2/4 [00:02<00:02,  1.45s/it]

1/1 [==============================] - 3s 3s/step


 75%|███████▌  | 3/4 [00:05<00:02,  2.13s/it]

1/1 [==============================] - 1s 978ms/step


100%|██████████| 4/4 [00:09<00:00,  2.29s/it]
